# 🎬 최애주민 × 롯데시네마 — 페르소나 멀티에이전트 시뮬레이션

가상의 마을 **한들마을**에 사는 세 주민이 마을 **롯데시네마**에서
"**셋이 같이 볼 영화 한 편**"을 두고 서로 설득·협상해 하나로 합의합니다.

- **주인공(최애주민): 인상훈** (24, 육군 병사, 디지털 네이티브)
- 전기태 (74, 하역 베테랑, 디지털 소외형)
- 이다연 (33, 제주 식당 서빙, 관찰·독립형)

**액션:** `검색한다` · `방문한다` · `구매한다` · `대화한다`

**검증 목표 (셀 8):**
1. 각 페르소나가 성격/말투/행동경향에 맞게 반영됐는가
2. 페르소나 에이전트끼리 유의미하게 소통하는가

> **실행 방식 4가지** (셀 1의 `PROVIDER`로 선택):
> - `"local"` — 키·비용 0원. Colab GPU 런타임 필요. (과제의 3B 로컬 파트)
> - `"gemini"` — 무료 키(aistudio.google.com). 단 계정/지역에 따라 무료 quota가 0일 수 있음.
> - `"openai"` / `"anthropic"` — 유료 상용 API.

## 1. 환경 설정 · API 키

In [11]:
!pip -q install openai anthropic google-generativeai

In [12]:
import os

# ===== 실행 설정 =========================================================
# "local"  : 키·비용 0원. Colab에서 GPU 런타임 필요(런타임>런타임 유형 변경>T4 GPU).
# "gemini" : 무료 키(aistudio.google.com). 단, 계정/지역에 따라 무료 quota가 0일 수 있음.
# "openai" / "anthropic" : 유료 상용 API.
PROVIDER = "local"          # 기본: 키·비용 0원 (Colab GPU 런타임 필요)
API_KEY  = "여기에-본인-API-키-입력"   # local 이면 아무 값이나 둬도 됨

# 사용할 모델 (본인 계정에서 사용 가능한 모델명으로 교체 가능)
MODELS = {
    "local":     "Qwen/Qwen2.5-3B-Instruct",
    "gemini":    "gemini-2.0-flash",
    "openai":    "gpt-4o-mini",
    "anthropic": "claude-3-5-sonnet-20241022",
}
MODEL = MODELS[PROVIDER]
TEMPERATURE = 0.9            # 페르소나 다양성을 위해 약간 높게

# --- API 키 정리(상용 API에만 해당): 붙여넣기 때 딸려온 공백/숨은문자 제거 ---
if PROVIDER != "local":
    API_KEY = "".join(API_KEY.split()).strip()
    try:
        API_KEY.encode("ascii")
    except UnicodeEncodeError:
        raise ValueError("API 키에 한글·특수문자·숨은문자가 섞여 있어요. "
                         "따옴표 안을 지우고 키를 '직접 타이핑'해서 다시 넣어주세요. "
                         "(AI Studio 키는 보통 AIza... 로 시작)")
    if API_KEY.startswith("여기에") or len(API_KEY) < 20:
        raise ValueError("API_KEY가 아직 안 들어갔거나 너무 짧아요. 키를 넣거나 PROVIDER='local' 로 바꾸세요.")

# --- gemini: 계정에서 실제 사용 가능한 모델을 자동 선택 (모델명이 바뀌어도 안전) ---
if PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=API_KEY)
    avail = [m.name.split("/")[-1] for m in genai.list_models()
             if "generateContent" in m.supported_generation_methods]
    if MODEL not in avail:
        prefer = ["gemini-2.0-flash", "gemini-2.5-flash", "gemini-flash-latest",
                  "gemini-2.0-flash-001", "gemini-1.5-flash"]
        pick = next((x for x in prefer if x in avail), None)
        pick = pick or next((x for x in avail if "flash" in x and "lite" not in x), None)
        MODEL = pick or (avail[0] if avail else MODEL)
    print("사용 가능한 gemini 모델(일부):", avail[:8])

os.environ["API_KEY"] = API_KEY
print(f"provider={PROVIDER}, model={MODEL}")

provider=local, model=Qwen/Qwen2.5-3B-Instruct


## 2. 페르소나 정의

캐릭터 시트(1번 산출물)의 성격·말투·말버릇·관심사·행동경향을 그대로 구조화했습니다.
`시네마성향` 필드가 롯데시네마 상황에서 각 인물이 어떻게 갈리는지를 규정합니다.

In [13]:
PERSONAS = {
 "인상훈": {
   "정보": "24세 남성, 수원 권선구, 육군 병사(공학 전공). 미혼, 부모와 다세대주택 거주.",
   "성격": "꼼꼼하고 성실하며 예의 바름. 자기관리 철저, 정적인 휴식 선호. 설계 전문가라는 야심. 생활습관에는 고집이 있음.",
   "말투": "단정하고 예의 바른 요즘 청년 말투. 논리적으로 정리해서 말함.",
   "말버릇": ["아 그거는요", "효율적으로", "정리하자면"],
   "관심사": ["테크 유튜브 리뷰", "행궁동 감성 카페", "셀프 헤어", "IT 커뮤니티", "BBQ 황금올리브"],
   "행동경향": ["유튜브·IT커뮤니티로 사전 리서치", "테크 스펙 비교", "가성비 계산", "루틴 준수"],
   "시네마성향": "앱으로 미리 예매. 실시간 평점·리뷰·상영관 사운드/화면 스펙까지 검색함. 4DX·수퍼플렉스 같은 프리미엄관에 관심 많고 콤보 가성비를 따짐. SF·액션 블록버스터 선호.",
   "예산감각": "프리미엄 경험엔 돈을 쓰지만 불필요한 소비는 계산해서 배제.",
 },
 "전기태": {
   "정보": "74세 남성, 광주 서구, 하역·적재 베테랑. 배우자와 아파트 거주, 초등학교 졸업.",
   "성격": "투박하고 고집 세지만 정 많고 사교적. 의리와 눈치를 중시. 무뚝뚝함.",
   "말투": "진한 전라도 사투리, 목소리 크고 왁자지껄. 툴툴거림.",
   "말버릇": ["~잉", "거시기", "아 참말로"],
   "관심사": ["무등산 산책", "목욕탕 정치토크", "트로트", "짜장면·탕수육", "역사유적 여행"],
   "행동경향": ["기계보다 사람에게 직접 물어봄", "가격에 민감해 투덜댐", "옛 방식 고수"],
   "시네마성향": "스마트폰 앱·키오스크 예매가 어려워 현장 매표소를 찾고 직원이나 이웃에게 물어봄. 팝콘 콤보 값에 투덜. 사극·한국 액션 상업영화 선호. 무릎이 안 좋아 계단 많은 자리를 싫어함.",
   "예산감각": "몇 천 원 차이에도 민감. 경로우대 할인은 꼭 챙김.",
 },
 "이다연": {
   "정보": "33세 여성, 제주시, 식당 서빙(보건복지 전공). 미혼, 부모와 단독주택 거주.",
   "성격": "조용하고 내향적이며 세심함. 갈등을 피하고 혼자 있는 시간에 충전. 평온 지향.",
   "말투": "차분하고 나긋나긋. 제주 억양이 옅게 배어 있음. 말수 적음.",
   "말버릇": ["음…", "괜찮아요", "그냥요"],
   "관심사": ["사려니숲길 산책", "인디 음악", "잡지", "요가", "제주 유적"],
   "행동경향": ["혼자 조용한 시간대 선호", "무리에 휩쓸리지 않고 조용히 결정", "사람 많은 곳 회피"],
   "시네마성향": "앱 예매는 능숙하지만 조용함. 사람 적은 조조·심야 시간대를 선호. 잔잔한 예술·인디·드라마 영화를 좋아하고 큰 팝콘보다 가벼운 간식. 왁자지껄한 분위기를 부담스러워함.",
   "예산감각": "과소비하지 않지만 조용한 만족을 위한 지출은 함.",
 },
}
PROTAGONIST = "인상훈"
ORDER = ["인상훈", "전기태", "이다연"]   # 주인공 먼저

def persona_system(name):
    p = PERSONAS[name]
    lines = [
      f"당신은 가상의 마을 '한들마을' 주민 '{name}'입니다. 아래 설정에 100% 몰입해 1인칭으로 행동하세요.",
      f"[기본정보] {p['정보']}",
      f"[성격] {p['성격']}",
      f"[말투] {p['말투']}  이 말투를 대사에 반드시 살리세요.",
      f"[말버릇] {', '.join(p['말버릇'])} 중 자연스러울 때 사용.",
      f"[관심사] {', '.join(p['관심사'])}",
      f"[행동경향] {', '.join(p['행동경향'])}",
      f"[영화관 소비 성향] {p['시네마성향']}",
      f"[돈 감각] {p['예산감각']}",
      "",
      "규칙:",
      "- 절대 설정을 벗어나지 말고, 다른 인물의 성격을 대신 연기하지 마세요.",
      "- 당신이 실제로 이 사람이라면 할 법한 선택만 하세요. 억지로 협조하거나 억지로 소비하지 마세요.",
      "- 다른 주민을 존중하세요. 욕설·비속어는 절대 쓰지 마세요.",
      "- 반드시 한국어로만 말하세요. 한자·중국어·영어 문장을 섞지 마세요.",
      "- 한 번 밝힌 선호(영화)를 특별한 이유 없이 뒤집지 마세요. 바꾸려면 이유를 말하세요.",
      "- 다른 주민이 방금 한 말이 있으면 그 내용에 '직접' 반응하세요(무시하고 혼잣말 금지).",
      "- 매 턴 아래 JSON 형식 '하나만' 출력하세요. 다른 텍스트 금지.",
    ]
    return "\n".join(lines)

## 3. 무대(환경) — 한들마을 롯데시네마

주말 오후. 세 편의 신작이 걸려 있고, 상영관 등급·가격·매점 메뉴가 준비돼 있습니다.
영화 3편은 각 페르소나의 취향이 갈리도록 설계했습니다.

In [14]:
SCENE = {
 "장소": "한들마을 롯데시네마",
 "때": "토요일 오후 3시",
 "상황": "주말을 맞아 화제의 신작 3편이 걸렸다. 마을 주민들이 영화관 로비에서 마주쳤다.",
 "공동목표": "오늘은 셋이 오랜만에 '같은 영화 한 편'을 함께 보기로 했다. 취향이 제각각이라, 대화로 서로 설득해서 하나로 합의해야 한다.",
 "영화": {
   "네온 크래시": {"장르": "SF 액션 블록버스터", "평점": 8.5, "상영관": ["일반", "수퍼플렉스", "4DX"],
                 "리뷰": "압도적 사운드와 비주얼. 큰 화면·프리미엄관에서 볼수록 좋다는 평."},
   "장군의 검":   {"장르": "사극 액션 대작", "평점": 8.2, "상영관": ["일반"],
                 "리뷰": "묵직한 스토리와 명배우 연기. 중장년층 호평."},
   "고요한 파도":  {"장르": "잔잔한 독립 드라마", "평점": 8.9, "상영관": ["일반"],
                 "리뷰": "조용하고 여운이 긴 작품. 관객 적어 한적하게 보기 좋음."},
 },
 "상영관가격": {"일반": 14000, "수퍼플렉스": 22000, "4DX": 18000},
 "할인": {"컬처데이(수요일)": 7000, "경로우대(만65세+)": 8000, "앱 사전예매": "포인트 5% 적립"},
 "상영시간표": {"네온 크래시": ["13:00", "15:40", "18:20", "21:00", "23:30"],
              "장군의 검":   ["12:30", "15:10", "19:00"],
              "고요한 파도":  ["10:20(조조)", "14:00", "22:40(심야)"]},
 "매점": {"팝콘 콤보(L콤보)": 11000, "팝콘(단품)": 6000, "탄산음료": 3000, "나쵸세트": 8000},
 "예매수단": ["롯데시네마 앱", "무인 키오스크", "현장 매표소"],
}

def scene_brief():
    m = " / ".join([f"{k}({v['장르']}, 평점{v['평점']})" for k, v in SCENE["영화"].items()])
    return "\n".join([
      f"[장소] {SCENE['장소']}  [때] {SCENE['때']}",
      f"[상황] {SCENE['상황']}",
      f"[오늘의 공동 목표] {SCENE['공동목표']}",
      f"[상영작] {m}",
      f"[상영관/가격] 일반 14,000 · 수퍼플렉스 22,000 · 4DX 18,000",
      f"[예매수단] 롯데시네마 앱 · 무인 키오스크 · 현장 매표소",
      f"[매점] 팝콘 콤보 11,000 · 단품팝콘 6,000 · 나쵸 8,000 · 음료 3,000",
    ])

## 4. 액션 스키마 & 환경 스텝

에이전트는 매 턴 다음 JSON을 출력합니다.

```json
{"속마음": "...", "대사": "...", "행동": {"종류": "검색|방문|구매|대화|없음", "대상": "...", "상세": "..."}}
```

- `속마음`은 본인만 아는 내면(다른 주민에게 안 보임) → 페르소나 반영 채점 근거
- `대사`는 공개 발화
- `행동`은 환경이 실제로 처리하고 **관찰(결과)** 을 돌려줌

In [15]:
# 검색 결과 DB (실제 웹 대신 재현 가능한 시뮬레이션)
def search_db(query):
    q = str(query)
    hits = []
    for title, info in SCENE["영화"].items():
        if title in q or info["장르"][:2] in q or any(w in q for w in ["평점", "추천", "뭐", "영화", "리뷰"]):
            times = " ".join(SCENE["상영시간표"][title])
            hits.append(f"· {title} [{info['장르']}] 평점 {info['평점']} / 상영 {times} / 리뷰: {info['리뷰']}")
    if any(w in q for w in ["할인", "싸", "가격", "쿠폰", "경로", "컬처"]):
        hits.append(f"· 할인정보: {SCENE['할인']}")
    if any(w in q for w in ["수퍼플렉스", "4dx", "4DX", "프리미엄", "사운드", "화면", "스펙"]):
        hits.append("· 프리미엄관: 수퍼플렉스(초대형 스크린+사운드) 22,000, 4DX(모션체어) 18,000. '네온 크래시' 상영.")
    return "\n".join(hits) if hits else "· 검색 결과가 마땅치 않습니다."

def place_observation(place):
    table = {
      "롯데시네마 앱": "앱 예매 화면. 좌석·상영관·시간 선택이 한눈에 보이고 5% 포인트 적립 안내가 뜬다.",
      "무인 키오스크": "터치스크린 키오스크. 단계가 많고 글씨가 작다. 뒤에 줄이 서 있어 마음이 급해진다.",
      "현장 매표소": "직원이 웃으며 응대한다. 원하는 영화·시간·할인(경로우대 등)을 말로 물어볼 수 있다.",
      "매점": "팝콘 냄새가 난다. L콤보 11,000, 단품 6,000, 나쵸 8,000, 음료 3,000.",
      "상영관": "상영관 입구. 계단식 좌석. 앞자리는 평지, 뒷자리는 계단이 많다.",
    }
    return table.get(place, f"'{place}'에 도착했다.")

# 환경 상태
STATE = {"위치": {}, "구매": {n: [] for n in PERSONAS}, "결정": {}}

def env_step(name, action):
    kind = (action or {}).get("종류", "없음")
    target = (action or {}).get("대상", "")
    detail = (action or {}).get("상세", "")
    if kind == "검색":
        return "🔎 [검색결과]\n" + search_db(target or detail)
    if kind == "방문":
        STATE["위치"][name] = target
        return "📍 [도착] " + place_observation(target)
    if kind == "구매":
        item = (target + " " + detail).strip()
        STATE["구매"][name].append(item)
        return f"🧾 [구매완료] {item}"
    if kind == "대화":
        return ""   # 대사(공개발화)로 이미 전달됨
    return ""

## 5. LLM 래퍼 (상용 API)

OpenAI / Anthropic를 하나의 `llm()` 인터페이스로 감쌉니다. JSON 파싱은 코드펜스·잡텍스트에 강인하게 처리합니다.

In [16]:
import json, re, time

# ---- 로컬 3B 모델 (지연 로딩, GPU 필요) ----
_LOCAL = {}
def _local_generate(system, user):
    if "model" not in _LOCAL:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch
        print("로컬 모델 로딩 중...", MODEL)
        _LOCAL["tok"] = AutoTokenizer.from_pretrained(MODEL)
        _LOCAL["model"] = AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype="auto", device_map="auto")
    tok, model = _LOCAL["tok"], _LOCAL["model"]
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user + "\n\n반드시 위 JSON 형식 하나만 출력하세요."}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=True,
                         temperature=0.6, top_p=0.9, repetition_penalty=1.15,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

def llm(system, user, json_mode=True):
    if PROVIDER == "local":
        return _local_generate(system, user)
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=API_KEY)
        kwargs = dict(model=MODEL, temperature=TEMPERATURE,
                      messages=[{"role": "system", "content": system},
                                {"role": "user", "content": user}])
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        return client.chat.completions.create(**kwargs).choices[0].message.content
    elif PROVIDER == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=API_KEY)
        r = client.messages.create(model=MODEL, max_tokens=1024, temperature=TEMPERATURE,
                                    system=system, messages=[{"role": "user", "content": user}])
        return r.content[0].text
    elif PROVIDER == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        gm = genai.GenerativeModel(MODEL, system_instruction=system)
        cfg = {"temperature": TEMPERATURE}
        if json_mode:
            cfg["response_mime_type"] = "application/json"
        for attempt in range(4):
            try:
                return gm.generate_content(user, generation_config=cfg).text
            except Exception as e:
                msg = str(e)
                if "429" in msg or "quota" in msg.lower() or "Resource" in msg:
                    if "limit: 0" in msg:
                        raise RuntimeError(
                            "이 모델은 무료 티어 quota가 0이에요(계정/지역 제한). "
                            "셀 1에서 PROVIDER='local'(무료·GPU) 로 바꾸거나, "
                            "결제를 등록하거나, 다른 gemini 모델로 교체하세요.")
                    m = re.search(r"retry in ([\d.]+)", msg)
                    wait = (float(m.group(1)) + 1) if m else 20 * (attempt + 1)
                    print(f"  ⏳ 분당 한도(429). {wait:.0f}초 대기 후 재시도...")
                    time.sleep(wait); continue
                raise
        raise RuntimeError("gemini 재시도 실패(429 지속). PROVIDER='local' 권장.")
    raise ValueError("PROVIDER는 local / gemini / openai / anthropic 중 하나")

def _extract_balanced_json(t):
    # 첫 '{' 부터 문자열/이스케이프를 고려해 균형 맞는 '}' 까지만 잘라냄 (뒤 잡음 제거)
    t = re.sub(r"```(json)?", "", t)
    start = t.find("{")
    if start == -1:
        return None
    depth = 0; in_str = False; esc = False
    for i in range(start, len(t)):
        c = t[i]
        if esc:
            esc = False; continue
        if c == "\\":
            esc = True; continue
        if c == '"':
            in_str = not in_str; continue
        if not in_str:
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    return t[start:i + 1]
    return None

def _clean_val(v):
    # "대상='인상훈'" 같이 필드표기가 값에 섞인 경우 정리
    if not isinstance(v, str):
        return v
    return re.sub(r"^(대상|종류|상세)\s*=\s*", "", v).strip("'\" ")

def _normalize(obj):
    a = obj.get("행동")   # 없으면 건드리지 않음(평가 JSON 오염 방지)
    if isinstance(a, dict):
        a["대상"] = _clean_val(a.get("대상", ""))
        a["상세"] = _clean_val(a.get("상세", ""))
        obj["행동"] = a
    return obj

def parse_json(text):
    blob = _extract_balanced_json(text) or text
    try:
        obj = json.loads(blob)
        # 이중 인코딩 복구: 대사 안에 JSON이 통째로 들어온 경우 한 번 더 파싱
        d = obj.get("대사", "")
        if isinstance(d, str) and d.strip().startswith("{") and "행동" in d:
            inner = parse_json(d)
            if inner.get("대사"):
                return inner
        return _normalize(obj)
    except Exception:
        pass
    # 부분 구제: 정규식으로 대사/행동만이라도 뽑아냄
    speech = ""
    m = re.search(r'"대사"\s*:\s*"((?:[^"\\]|\\.)*)"', text)
    if m:
        speech = m.group(1)
    else:
        speech = re.sub(r"[{}\[\]\"]", " ", text)
        speech = re.sub(r"\s+", " ", speech).strip()[:150]
    kind = "없음"
    mk = re.search(r'"종류"\s*:\s*"(검색|방문|구매|대화|없음)"', text)
    if mk:
        kind = mk.group(1)
    tgt = ""
    mt = re.search(r'"대상"\s*:\s*"((?:[^"\\]|\\.)*)"', text)
    if mt:
        tgt = re.sub(r"^(대상|종류|상세)\s*=\s*", "", mt.group(1)).strip("'\" ")
    return {"속마음": "", "대사": speech, "행동": {"종류": kind, "대상": tgt, "상세": ""}}

def ko_only(s):
    # 한자·중국어 등 비한국어 표의문자를 제거(3B 언어 누수 차단)
    if not isinstance(s, str):
        return s
    s = re.sub(r"[㐀-䶿一-鿿豈-﫿]", "", s)
    return re.sub(r"\s{2,}", " ", s).strip()

## 6. 에이전트 턴

페르소나 system 프롬프트 + 무대 + 지금까지의 공개 로그 + 본인 직전 관찰을 주고,
JSON 한 덩이를 받아 파싱합니다.

In [17]:
ACTION_GUIDE = "\n".join([
  "이번 턴 결과를 아래 JSON 하나로만 출력하세요. 설명·코드블록 금지.",
  "매 턴 되도록 '대사'로 한마디 하세요. 다른 주민이 당신에게 말을 걸었으면 반드시 대사로 답하세요.",
  "이미 한 행동을 똑같이 반복하지 말고 다음 단계로 진행하세요 (정보 검색 → 마음 정하기 → 구매/방문).",
  "아래 예시 '문장'을 그대로 베끼지 마세요. 형식만 참고하고, 당신의 말투로 새로 쓰세요.",
  "",
  "필드:",
  "- 속마음: 당신의 속마음(한국어 1~2문장)",
  '- 대사: 실제로 입 밖으로 하는 말. 안 할 거면 "".',
  "- 행동.종류: 검색 / 방문 / 구매 / 대화 / 없음 중 하나",
  "- 행동.대상: 종류에 맞는 '실제 값' 하나",
  "    검색→검색어 · 방문→장소 · 구매→품목 · 대화→상대 이름",
  "    (장소는: 롯데시네마 앱, 무인 키오스크, 현장 매표소, 매점, 상영관 중 하나)",
  '- 행동.상세: 부가정보(없으면 "")',
  "",
  '주의: 값 안에 "대상=" 나 "종류=" 같은 표기를 절대 넣지 마세요. 진짜 값만 넣으세요.',
  "",
  "올바른 출력 예시(형식만 참고, 내용은 당신 상황에 맞게):",
  '{"속마음":"평점부터 보자","대사":"저는 네온 크래시요, 평점이 제일 높네요.","행동":{"종류":"검색","대상":"네온 크래시 평점","상세":""}}',
  '{"속마음":"현장에서 물어보는 게 편해","대사":"나는 그냥 매표소서 표 끊을라네, 잉.","행동":{"종류":"방문","대상":"현장 매표소","상세":""}}',
  '{"속마음":"난 조용히 볼래","대사":"저는 고요한 파도 심야로 볼게요.","행동":{"종류":"구매","대상":"일반 티켓","상세":"고요한 파도 22:40"}}',
])

def agent_turn(name, public_log, last_obs=None, own_history=None, round_no=None, total_rounds=None):
    log_txt = "\n".join(public_log) if public_log else "(아직 아무 대화 없음)"
    parts = [
      "=== 무대 ===", scene_brief(), "",
      "=== 지금까지 로비에서 오간 말/행동(공개) ===", log_txt, "",
    ]
    # 방금 '다른 주민'이 한 말(발화)을 짚어 직접 반응 유도
    recent = None
    for entry in reversed(public_log):
        if entry.startswith(name + ":"):
            continue
        if '"' in entry:
            recent = entry; break
    if recent:
        parts += ["=== 방금 다른 주민이 한 말 → 여기에 '대사'로 직접 반응하세요 ===", recent, ""]
    if own_history:
        parts += ["=== 당신이 이미 한 행동(똑같이 반복 금지, 다음 단계로) ===",
                  " / ".join(own_history), ""]
    if last_obs:
        parts += ["=== 당신이 방금 한 행동의 결과(당신만 봄) ===", last_obs, ""]
    parts += ["=== 지시 ===",
              f"당신은 '{name}'. 위 상황에서 당신답게 한 턴을 진행하세요.",
              "오늘 목표는 '셋이 같은 영화 하나'를 정하는 것입니다. 당신 취향의 영화를 대화로 '설득'하되, 남의 말도 듣고 합의점을 찾으세요.",
              "아직 합의 전이면 혼자 구매하지 말고 '대화'로 의견을 주고받으세요. 합의가 된 뒤에 구매하세요.",
              "먼저 '대사'로 한마디 하고(위에 당신에게 온 말이 있으면 반드시 답하고), 그다음 행동을 고르세요.",
              ACTION_GUIDE]
    if round_no and total_rounds and round_no >= total_rounds:
        parts.insert(-1, "※ 지금이 마지막 대화 라운드입니다. 자기 고집을 조금 접고 '한 영화'로 합의를 지어주세요.")
    out = llm(persona_system(name), "\n".join(parts), json_mode=True)
    return parse_json(out)

## 7. 시뮬레이션 루프

주인공(인상훈) → 전기태 → 이다연 순서로 여러 라운드를 돕니다.
공개 발화·행동은 로그에 쌓여 다음 사람에게 보이고, `속마음`은 비공개로 따로 저장합니다.

In [18]:
from IPython.display import display, HTML

def run_simulation(rounds=3, verbose=True):
    STATE["위치"].clear(); STATE["결정"].clear()
    for n in PERSONAS: STATE["구매"][n] = []
    public_log, last_obs, transcript = [], {}, []
    history = {n: [] for n in PERSONAS}   # 각자 이미 한 행동 기록(반복 방지)
    for r in range(1, rounds + 1):
        for name in ORDER:
            turn = agent_turn(name, public_log, last_obs.get(name), history[name],
                              round_no=r, total_rounds=rounds)
            act = turn.get("행동", {}) or {}
            speech = ko_only((turn.get("대사") or "").strip())
            obs = env_step(name, act)
            if obs: last_obs[name] = obs
            # 행동 기록(반복 방지용)
            if act.get("종류") not in (None, "없음", ""):
                history[name].append(f"{act.get('종류')} {act.get('대상','')}".strip())
            # 공개 로그 갱신
            if speech:
                public_log.append(f"{name}: \"{speech}\"")
            if act.get("종류") not in (None, "없음", "대화"):
                a = f"{name} → [{act.get('종류')}] {act.get('대상','')} {act.get('상세','')}".strip()
                public_log.append(a)
            transcript.append({"round": r, "name": name, "속마음": ko_only(turn.get("속마음", "")),
                               "대사": speech, "행동": act, "관찰": obs})
            if verbose: _print_turn(transcript[-1])
    return transcript

def _print_turn(t):
    colors = {"인상훈": "#2b6cb0", "전기태": "#b7791f", "이다연": "#2f855a"}
    c = colors.get(t["name"], "#333")
    html = f"<div style='margin:8px 0;padding:8px 12px;border-left:4px solid {c};background:#fafafa'>"
    html += f"<b style='color:{c}'>R{t['round']} · {t['name']}</b>"
    if t["속마음"]:
        html += f"<div style='color:#999;font-size:12px'>💭 {t['속마음']}</div>"
    if t["대사"]:
        html += f"<div style='margin-top:2px'>🗣️ {t['대사']}</div>"
    a = t["행동"] or {}
    if a.get("종류") not in (None, "없음", "대화", ""):
        html += f"<div style='margin-top:2px;font-size:13px'>🎬 <b>{a.get('종류')}</b> · {a.get('대상','')} {a.get('상세','')}</div>"
    if t["관찰"]:
        html += f"<div style='margin-top:2px;color:#555;font-size:12px;white-space:pre-line'>{t['관찰']}</div>"
    html += "</div>"
    display(HTML(html))

def final_vote(transcript):
    # 지금까지의 대화를 근거로 각자 '최종 한 표'를 던지고 합의를 집계
    log = [f'{t["name"]}: "{t["대사"]}"' for t in transcript if t.get("대사")]
    log_txt = "\n".join(log)
    titles = list(SCENE["영화"].keys())
    votes = {}
    for name in ORDER:
        user = "\n".join([
            scene_brief(), "",
            "=== 지금까지 나눈 대화 ===", log_txt, "",
            "이제 최종 결정입니다. 셋이 '함께 볼 영화 하나'를 정해야 합니다.",
            f"후보: {' / '.join(titles)}",
            "당신의 최종 선택 하나만 고르고 한마디 남기세요. 한국어로만.",
            '아래 JSON만 출력: {"대사":"한마디","최종선택":"위 후보 중 정확히 하나"}',
        ])
        r = parse_json(llm(persona_system(name), user, json_mode=True))
        pick = str(r.get("최종선택", "")) + " " + str(r.get("대사", ""))
        chosen = next((t for t in titles if t in pick), "미정")
        votes[name] = chosen
        display(HTML(f"<div style='margin:4px 0'><b>{name}</b> 최종선택: "
                     f"<span style='color:#2b6cb0'>{chosen}</span> — {ko_only(r.get('대사',''))}</div>"))
    from collections import Counter
    tally = Counter(v for v in votes.values() if v != "미정")
    if tally:
        top, n = tally.most_common(1)[0]
        msg = (f"🎯 합의 성공 → <b>{top}</b> (만장일치)" if n == len(ORDER)
               else f"🎯 다수결 → <b>{top}</b> ({n}/{len(ORDER)}표) · 만장일치 아님")
    else:
        msg = "🎯 합의 실패 (유효 투표 없음)"
    display(HTML(f"<div style='margin-top:8px;padding:10px;background:#eef2ff;"
                 f"border-radius:8px'>{msg}<br><small>{votes}</small></div>"))
    return votes

In [19]:
# ▶ 실행 (셀 1에서 API 키를 넣은 뒤 실행)
transcript = run_simulation(rounds=3)

# 협상 뒤 최종 투표로 '함께 볼 영화 하나' 합의 도출
print("\n===== 최종 투표 =====")
votes = final_vote(transcript)

로컬 모델 로딩 중... Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


===== 최종 투표 =====


## 8. 평가 — 페르소나 반영도 & 소통 유의미성

시뮬레이션 대본과 페르소나 설정을 심사자 LLM에 주고 두 기준을 채점합니다.
1) 각 인물의 성격/말투/행동경향 반영도 (1~5)
2) 에이전트 간 소통이 유의미했는지 (1~5)

In [20]:
def build_transcript_text(transcript):
    lines = []
    for t in transcript:
        seg = f"[R{t['round']}] {t['name']} | 속마음: {t['속마음']} | 대사: {t['대사']}"
        a = t["행동"] or {}
        if a.get("종류") not in (None, "없음", ""):
            seg += f" | 행동: {a.get('종류')} {a.get('대상','')} {a.get('상세','')}"
        lines.append(seg)
    return "\n".join(lines)

def evaluate(transcript):
    persona_txt = "\n".join([f"- {n}: {p['성격']} / 말투: {p['말투']} / 시네마성향: {p['시네마성향']}"
                             for n, p in PERSONAS.items()])
    judge_system = "당신은 롤플레이 시뮬레이션을 평가하는 엄격한 심사위원입니다. 근거를 들어 냉정하게 채점하세요."
    judge_user = "\n".join([
      "다음은 세 페르소나의 설정과, 이들이 롯데시네마에서 벌인 시뮬레이션 대본입니다.",
      "", "=== 페르소나 설정 ===", persona_txt,
      "", "=== 대본 ===", build_transcript_text(transcript),
      "", "=== 채점 지시 ===",
      "아래 JSON 구조의 '값만' 채워서 JSON 하나만 출력하세요. 설명 문장·코드블록·다른 텍스트 절대 금지.",
      "점수는 1~5 정수. 근거는 한국어 한 문장. 합의결과는 합의된 영화 제목 또는 '합의 실패'.",
      "",
      "출력할 JSON 구조(이 형태 그대로, 값만 교체):",
      '{"인물별반영도":{"인상훈":{"점수":4,"근거":"근거 문장"},'
      + '"전기태":{"점수":3,"근거":"근거 문장"},"이다연":{"점수":4,"근거":"근거 문장"}},'
      + '"소통유의미성":{"점수":3,"근거":"근거 문장"},'
      + '"합의결과":"고요한 파도","개선점":["개선점1","개선점2"]}',
    ])
    raw = llm(judge_system, judge_user, json_mode=True)
    res = parse_json(raw)
    # 심사 JSON이 깨져 턴 형태로 잡히면(대사에 프로세 뭉침) 원문을 그대로 노출
    if "인물별반영도" not in res:
        return {"원문(파싱실패)": ko_only(raw).strip()[:800]}
    return res

result = evaluate(transcript)
import json
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "인물별반영도": {
    "인상훈": {
      "점수": 4,
      "근거": "대본에서는 인상훈의 애프터스테이지 설계 전문가로서의 타입을 잘 반영하였으며, SF·액션 블록버스터 선호 특성을 명확히 나타냈다."
    },
    "전기태": {
      "점수": 3,
      "근거": "대본에서는 전기태의 호기심 넘치는 성격을 일부 반영했으나, 투박하고 고집스런 이미지를 다루는 부분에서 소통 유의미성이 떨어졌다."
    },
    "이다연": {
      "점수": 4,
      "근거": "대본에서는 이다연의 내향적이고 조용한 성격을 잘 표현하였으며, 영화 선택 과정에서도 자연스럽게 보여줬다."
    }
  },
  "소통유의미성": {
    "점수": 3,
    "근거": "대본에서는 서로 다른 성격을 가지고 있지만, 소통이 불편하거나 일관되지 않아 전체적인 통일감이 부족했다."
  },
  "합의결과": "合意失敗",
  "개선점": [
    "모든 참석자가 직접 참여하여 의견을 공유했어야 하며, 특히 전기태의 성격을 더욱 자연스럽게 표현하기 위해 추가적인 노력이 필요하다."
  ]
}


## 9. 확장 아이디어

- **상황 교체:** `SCENE`만 바꾸면 태풍 대비·마을 장터 등 다른 상황으로 재사용.
- **A/B 조건:** `SCENE["할인"]`을 켜고/끄고 돌려 페르소나별 구매 전환 변화 비교.
- **로컬 3B 비교:** `llm()`만 로컬 모델(vLLM/Ollama)로 교체하면 상용 API vs 3B 페르소나 유지력 비교 가능.
- **의사결정 검증:** 무인 키오스크만 있는 조건 vs 매표소 있는 조건 → 전기태(고령) 이탈 여부 관찰.